# MPC Example

The below example is taken from [OSQP's documentation](https://osqp.org/docs/examples/mpc.html) to demonstrate how to use a simple MPC problem using `CuPIQP`.


In [1]:
import sys
sys.path.append('./')
sys.path.append('../')

import numpy as np
from scipy import sparse	


from cupiqp import SolverBase

%load_ext autoreload
%autoreload 2

In [2]:
INF = np.inf

# Discrete time model of a quadcopter
Ad = sparse.csc_matrix([
  [1.,      0.,     0., 0., 0., 0., 0.1,     0.,     0.,  0.,     0.,     0.    ],
  [0.,      1.,     0., 0., 0., 0., 0.,      0.1,    0.,  0.,     0.,     0.    ],
  [0.,      0.,     1., 0., 0., 0., 0.,      0.,     0.1, 0.,     0.,     0.    ],
  [0.0488,  0.,     0., 1., 0., 0., 0.0016,  0.,     0.,  0.0992, 0.,     0.    ],
  [0.,     -0.0488, 0., 0., 1., 0., 0.,     -0.0016, 0.,  0.,     0.0992, 0.    ],
  [0.,      0.,     0., 0., 0., 1., 0.,      0.,     0.,  0.,     0.,     0.0992],
  [0.,      0.,     0., 0., 0., 0., 1.,      0.,     0.,  0.,     0.,     0.    ],
  [0.,      0.,     0., 0., 0., 0., 0.,      1.,     0.,  0.,     0.,     0.    ],
  [0.,      0.,     0., 0., 0., 0., 0.,      0.,     1.,  0.,     0.,     0.    ],
  [0.9734,  0.,     0., 0., 0., 0., 0.0488,  0.,     0.,  0.9846, 0.,     0.    ],
  [0.,     -0.9734, 0., 0., 0., 0., 0.,     -0.0488, 0.,  0.,     0.9846, 0.    ],
  [0.,      0.,     0., 0., 0., 0., 0.,      0.,     0.,  0.,     0.,     0.9846]
])
Bd = sparse.csc_matrix([
  [0.,      -0.0726,  0.,     0.0726],
  [-0.0726,  0.,      0.0726, 0.    ],
  [-0.0152,  0.0152, -0.0152, 0.0152],
  [-0.,     -0.0006, -0.,     0.0006],
  [0.0006,   0.,     -0.0006, 0.0000],
  [0.0106,   0.0106,  0.0106, 0.0106],
  [0,       -1.4512,  0.,     1.4512],
  [-1.4512,  0.,      1.4512, 0.    ],
  [-0.3049,  0.3049, -0.3049, 0.3049],
  [-0.,     -0.0236,  0.,     0.0236],
  [0.0236,   0.,     -0.0236, 0.    ],
  [0.2107,   0.2107,  0.2107, 0.2107]])
[nx, nu] = Bd.shape

# Constraints
u0 = 10.5916
umin = np.array([9.6, 9.6, 9.6, 9.6]) - u0
umax = np.array([13., 13., 13., 13.]) - u0

xmin = np.array([-np.pi/6,-np.pi/6,-INF,-INF,-INF,-1.,
                 -INF,-INF,-INF,-INF,-INF,-INF])
xmax = np.array([ np.pi/6, np.pi/6, INF, INF, INF, INF,
                  INF, INF, INF, INF, INF, INF])

# Objective function
Q = sparse.diags([0., 0., 10., 10., 10., 10., 0., 0., 0., 5., 5., 5.])
QN = Q
R = 0.1*sparse.eye(4)

# Initial and reference states
x0 = np.zeros(12)
xr = np.array([0.,0.,1.,0.,0.,0.,0.,0.,0.,0.,0.,0.])

# Prediction horizon
N = 10

# Cast MPC problem to a QP: x = (x(0),x(1),...,x(N),u(0),...,u(N-1))
# - quadratic objective
P = sparse.block_diag([sparse.kron(sparse.eye(N), Q), QN,
                       sparse.kron(sparse.eye(N), R)], format='csc')
# - linear objective
q = np.hstack([np.kron(np.ones(N), -Q@xr), -QN@xr, np.zeros(N*nu)])
# - linear dynamics
Ax = sparse.kron(sparse.eye(N+1),-sparse.eye(nx)) + sparse.kron(sparse.eye(N+1, k=-1), Ad)
Bu = sparse.kron(sparse.vstack([sparse.csc_matrix((1, N)), sparse.eye(N)]), Bd)
Aeq = sparse.hstack([Ax, Bu])
leq = np.hstack([-x0, np.zeros(N*nx)])
ueq = leq
# - input and state constraints
Aineq = sparse.eye((N+1)*nx + N*nu)
lineq = np.hstack([np.kron(np.ones(N+1), xmin), np.kron(np.ones(N), umin)])
uineq = np.hstack([np.kron(np.ones(N+1), xmax), np.kron(np.ones(N), umax)])
# - OSQP constraints
A = sparse.vstack([Aeq, Aineq], format='csc')
l = np.hstack([leq, lineq])
u = np.hstack([ueq, uineq])

idx_l_inf = np.where(lineq <= -1e5)[0]
idx_u_inf = np.where(uineq >= 1e5)[0]
lineq[idx_l_inf] = -1e5 * np.ones_like(idx_l_inf)
uineq[idx_u_inf] = 1e5 * np.ones_like(idx_u_inf)

x_l = -INF * np.ones_like(q)
x_u = INF * np.ones_like(q)

x_u = 1e3 * np.ones_like(q)
x_l = -1e3 * np.ones_like(q)

Solve with OSQP:

In [3]:
import osqp
# Create an OSQP object
prob = osqp.OSQP()

# Setup workspace
prob.setup(P, q, A, l, u, warm_starting=True, verbose=True)
res = prob.solve()

print("Run time: ", res.info.run_time)

-----------------------------------------------------------------
           OSQP v1.0.0  -  Operator Splitting QP Solver
              (c) The OSQP Developer Team
-----------------------------------------------------------------
problem:  variables n = 172, constraints m = 304
          nnz(P) + nnz(A) = 1161
settings: algebra = Built-in,
          OSQPInt = 4 bytes, OSQPFloat = 8 bytes,
          linear system solver = QDLDL v0.1.8,
          eps_abs = 1.0e-03, eps_rel = 1.0e-03,
          eps_prim_inf = 1.0e-04, eps_dual_inf = 1.0e-04,
          rho = 1.00e-01 (adaptive: 50 iterations),
          sigma = 1.00e-06, alpha = 1.60, max_iter = 4000
          check_termination: on (interval 25, duality gap: on),
          time_limit: 1.00e+10 sec,
          scaling: on (10 iterations), scaled_termination: off
          warm starting: on, polishing: off, 
iter   objective    prim res   dual res   gap        rel kkt    rho         time
   1  -3.2965e+01   8.15e-01   6.00e+00   5.49e+01   5.

Solver with CuPIQP:

In [4]:
import cupy as cp
from cupyx.scipy.sparse import csr_matrix

solver = SolverBase()
solver.settings.kkt_solver = 'sparse_ldlt'
# solver.settings.debug = True
solver.settings.verbose = True
solver.settings.max_iter = 30
with cp.cuda.Device(0):
	solver.setup(
		P=csr_matrix(P),  # Convert scipy sparse to cupy sparse directly
		c=cp.array(q),
		A=csr_matrix(Aeq),  # Convert scipy sparse to cupy sparse directly
		b=cp.array(leq),
		G=csr_matrix(Aineq),  # Convert scipy sparse to cupy sparse directly
		h_u=cp.array(uineq),
		h_l=cp.array(lineq),
		x_u=cp.array(x_u),
		x_l=cp.array(x_l)
	)

	result = solver.solve()

print("status: ", solver._result.info.status)

	

No multithreading interface library was specified using the DirectSolverOptions. The performance of CPU operations like planning will be significantly lower than if you provide a multithreading library.


sparse backend:
variables n = 172, nnz(P) = 117
equality constraints p = 132, nnz(A) = 872
inequality constraints m = 172, nnz(G) = 172
inequality lower bounds n_h_l = 172
inequality upper bounds n_h_u = 172
variable lower bounds n_x_l = 172
variable upper bounds n_x_u = 172

iter  prim_obj       dual_obj       duality_gap   prim_res      dual_res      rho         delta       mu          p_step   d_step
Warp 1.11.0.dev20251115 initialized:
   Git commit: 62778e81d556f6573509dc314a49794370940d55
   CUDA Toolkit 12.8, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce RTX 5090" (31 GiB, sm_120, mempool enabled)
   Kernel cache:
     /home/fenglong/.cache/warp/1.11.0.dev20251115
Module cupiqp.kkt_systems a431728 load on device 'cuda:0' took 0.49 ms  (cached)
  0    4.83067613e+08   -9.57664549e+10   9.62495225e+10   2.27651175e+04   8.30398095e+04   1.000000e-06   1.000000e-04   4.395100e+08   0.0000000   0.0000000
Solver::_calculate_step capturing CUDA g